# ST7 Project 2026

## Algorithm
1. Forward solve: m_n → SEM3D → u_sim(x_ric, t) and u(x,t)
2. Misfit: r(t) = u_sim(t) - d_obs(t)
3. Adjoint solve: r(T-t) → SEM3D backward → Λ(x,t)
4. Gradient: g_λ, g_μ from cross-correlation of ε[u] and ε[Λ]
5. CG direction (Fletcher-Reeves): p_n = -g_n + β_n · p_{n-1}
6. Line search (backtracking Armijo): find α_n
7. Update: m_{n+1} = m_n + α_n · p_n

## 0. Imports

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import os
import sys
import h5py
# from pysem import parse_sem3d_traces
from pathlib import Path

# sys.path.append(str(Path("pysem/src").resolve()))

path_to_src = str(Path("pysem/src").resolve())
print(f"Adding {path_to_src} to sys.path")
if path_to_src not in sys.path:
    sys.path.append(path_to_src)

from pysem.parse_sem3d_traces import ParseSEM3DH5Traces
from util_funct.sbatch_and_wait import sbatch_and_wait
from util_funct.compute_misfit import compute_misfit
from util_funct.write_backward_spec import write_backward_spec_from_template
from util_funct.write_misfit_files import write_time_reversed_residual_files
from util_funct.load_global_xyz_tuples import load_global_xyz_tuples
from gradient_search_direction.util_funct.update_parameters_file import update_parameters_file
from gradient_search_direction.util_funct.modify_h5 import modify_h5_g

import time
from util_to_be_deleted.inspect_h5_samples import inspect_h5_samples

Adding /usr/users/cea_seism/benede_gio/CEISM-Project/pysem/src to sys.path


## 1. Paths and parameters

In [2]:
SEM3D_CONFIG_RES_FOLDER_PATH = "./sem3d_config_files"

FORWARD_PROBLEM_MESHER_SBATCH_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "MESHER.sbatch")
FORWARD_PROBLEM_SOLVER_SBATCH_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "SOLVER.sbatch")

TRACES_SIMULATED_FOLDER_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "traces")
TRACES_OSSERVATED_FOLDER_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "Uobs")

SEM3D_CONFIG_RES_FOLDER_PATH_ADJ = "./sem3d_config_files_adj"

#ADJOINT_SOURCES_FOLDER_NAME = ""    #MUST BE short, otherwise sem3d will complain
ADJOINT_SOURCES_FOLDER_PATH = SEM3D_CONFIG_RES_FOLDER_PATH_ADJ

STATIONS_FILE_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ, "stations.txt")

BACKWARD_INPUT_SPEC_TEMPLATE_FILE_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "template/input_backward_template.spec")
ADJOINT_PROBLEM_SOLVER_SBATCH_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ, "SOLVER_ADJOINT.sbatch")

In [3]:
GRADIENT_SEARCH_DIRECTION_FOLDER_PATH = "./gradient_search_direction"

GRADIENT_SEARCH_DIRECTION_SBATCH_PATH = os.path.join(GRADIENT_SEARCH_DIRECTION_FOLDER_PATH, "gradient_search_direction.sbatch")
GRADIENT_SEARCH_DIRECTION_PARAMETERS_SBATCH_PATH = os.path.join(GRADIENT_SEARCH_DIRECTION_FOLDER_PATH, "gradient_search_parameters_file.json")

LBFGS_STATE_FOLDER_PATH = "state"
LBFGS_OUTPUT_FOLDER_PATH = "outputs"


In [4]:
N_ITER = 10 # to be modified
LBFGS_MEM = 5 # Numero di iterazioni da ricordare


## 3. Initial material m_0

In [5]:
#! python3 ./pysem/src/pysem/generate_h5_materials.py @@prop "la" "mu" "ds" @@tag "linear_gradient" @@dir "z" @@xlim -1300 1300 @@ylim -1300 1300 @@zlim -1540 0 @@step 13 13 77 @@pfx 'example'
#! mv example* {SEM3D_CONFIG_RES_FOLDER_PATH} 
#! cp  {SEM3D_CONFIG_RES_FOLDER_PATH}/example* {SEM3D_CONFIG_RES_FOLDER_PATH_ADJ}

! cp  ./materials_samples/example* {SEM3D_CONFIG_RES_FOLDER_PATH}
! cp  ./materials_samples/example* {SEM3D_CONFIG_RES_FOLDER_PATH_ADJ}

In [6]:
! rm -r ./sem3d_config_files/mat
! rm -r ./sem3d_config_files/res
! rm -r ./sem3d_config_files/traces
! rm -r ./sem3d_config_files/mirror
! rm -r ./sem3d_config_files/prot
#! rm -r ./sem3d_config_files/sem

! rm -r ./sem3d_config_files_adj/mat
! rm -r ./sem3d_config_files_adj/res
! rm -r ./sem3d_config_files_adj/traces
! rm -r ./sem3d_config_files_adj/mirror
! rm -r ./sem3d_config_files_adj/prot
#! rm -r ./sem3d_config_files_adj/sem

! rm -r ./gradient_search_direction/outputs/
! rm -r ./gradient_search_direction/state/
! rm ./gradient_search_direction/error.*
! rm ./gradient_search_direction/outputs/global_xyz_tuples.pkl
! rm ./gradient_search_direction/outputs/iter_*
! rm ./gradient_search_direction/mesh_mapping*
! rm ./gradient_search_direction/multi_hexahedral_mesh_*


rm: impossible de supprimer './gradient_search_direction/outputs/global_xyz_tuples.pkl': Aucun fichier ou dossier de ce nom
rm: impossible de supprimer './gradient_search_direction/outputs/iter_*': Aucun fichier ou dossier de ce nom


## 4. Algorithm

In [ ]:
N_ITER = 1 #to be modified

#sbatch_and_wait(FORWARD_PROBLEM_MESHER_SBATCH_PATH)
#! cp -r {SEM3D_CONFIG_RES_FOLDER_PATH}/sem {SEM3D_CONFIG_RES_FOLDER_PATH_ADJ}

stations = np.loadtxt(STATIONS_FILE_PATH) #read_stations_pos(STATIONS_FILE_PATH)

for n in range(N_ITER):

    # ── STEP 1 ────────────────────────────────────────
    
    sbatch_and_wait(FORWARD_PROBLEM_SOLVER_SBATCH_PATH)

    # ── STEP 2: MISFIT ────────────────────────────────────────────────────

    J, residual, t_sim, dt_sim = compute_misfit(TRACES_SIMULATED_FOLDER_PATH, TRACES_OSSERVATED_FOLDER_PATH)
      

    # ── STEP 3 ────────────────────────────────────────
    
    time_reversed_residual = residual[::-1, :, :]
    
    file_names = write_time_reversed_residual_files(time_reversed_residual, 
                                                  t_sim, 
                                                  OUTPUT_DIR=ADJOINT_SOURCES_FOLDER_PATH)

    write_backward_spec_from_template(template_backward_spec_path=BACKWARD_INPUT_SPEC_TEMPLATE_FILE_PATH, 
                                      output_backward_spec_path = SEM3D_CONFIG_RES_FOLDER_PATH_ADJ,
                                      adjoint_sources_folder_path = ADJOINT_SOURCES_FOLDER_PATH, 
                                      stations = stations, 
                                      file_names = file_names)
    
    sbatch_and_wait(ADJOINT_PROBLEM_SOLVER_SBATCH_PATH)
    
    # ── STEP 4-5: GRADIENT & SEARCH DIRECTIONS ─────────────────────────────────────────────────
    
    update_parameters_file(parameters_file_path=GRADIENT_SEARCH_DIRECTION_PARAMETERS_SBATCH_PATH, 
                           n_iter=n, 
                           LBFGS_MEM=LBFGS_MEM, 
                           LBFGS_STATE_FOLDER_PATH = LBFGS_STATE_FOLDER_PATH, 
                           LBFGS_OUTPUT_FOLDER_PATH = LBFGS_OUTPUT_FOLDER_PATH)
    
    sbatch_and_wait(GRADIENT_SEARCH_DIRECTION_SBATCH_PATH)



    # -----------------------------------------------------------------------
    # ESTRAZIONE DIREZIONI (r) E GRADIENTI (g) GLOBALI E ORDINATI
    # -----------------------------------------------------------------------
    iter_folder = f"iter_{str(n).zfill(4)}"    
    vectors_file_path = (
        Path(GRADIENT_SEARCH_DIRECTION_FOLDER_PATH).resolve()
        / LBFGS_OUTPUT_FOLDER_PATH
        / iter_folder
        / "global_vectors.npz"
    )

    max_tries = 30
    sleep_seconds = 2

    data = None
    last_err = None

    for _ in range(max_tries):
        if vectors_file_path.is_file():
            try:
                data = np.load(vectors_file_path)
                break
            except (OSError, EOFError, ValueError) as e:
                # file seen but maybe not fully visible / not fully flushed yet
                last_err = e
        time.sleep(sleep_seconds)

    if data is None:
        raise FileNotFoundError(
            f"Could not safely load file after waiting: {vectors_file_path}\n"
            f"Last error: {last_err}"
        )

    # Ecco le tue 4 variabili pronte, 1D, globali e ordinate secondo il mesh!
    dir_la = data['dir_lam']
    dir_mu  = data['dir_mu']
    p1, p2 = data['scalar_prod_lam'] , data['scalar_prod_mu']
    x_gl, y_gl, z_gl = data['x_gl'], data['y_gl'], data['z_gl']


Submitted MESHER.sbatch from /usr/users/cea_seism/benede_gio/CEISM-Project/sem3d_config_files -> job 182129
Job 182129 finished with state: COMPLETED
MESHER.sbatch executed in 63.14 seconds

Submitted SOLVER.sbatch from /usr/users/cea_seism/benede_gio/CEISM-Project/sem3d_config_files -> job 182130
Job 182130 finished with state: COMPLETED
SOLVER.sbatch executed in 93.18 seconds

Submitted SOLVER_ADJOINT.sbatch from /usr/users/cea_seism/benede_gio/CEISM-Project/sem3d_config_files_adj -> job 182132
Job 182132 finished with state: COMPLETED
SOLVER_ADJOINT.sbatch executed in 93.20 seconds

Submitted gradient_search_direction.sbatch from /usr/users/cea_seism/benede_gio/CEISM-Project/gradient_search_direction -> job 182133
Job 182133 finished with state: COMPLETED
gradient_search_direction.sbatch executed in 48.10 seconds



In [ ]:
import os
import glob
import shutil
import numpy as np
from pathlib import Path

# --- PATH MATERIALI ---
materials_paths = {
    'La': os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "example_la.h5"),
    'Mu': os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "example_mu.h5")
}

base_la_path = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "base_la_test_backup.h5")
base_mu_path = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "base_mu_test_backup.h5")

shutil.copy(materials_paths['La'], base_la_path)
shutil.copy(materials_paths['Mu'], base_mu_path)

mapping_list = [np.array([x_gl[i], y_gl[i], z_gl[i]]) for i in range(x_gl.size)]

# --- MISFIT BASE ---
J_base, _, _, _ = compute_misfit(
    TRACES_SIMULATED_FOLDER_PATH,
    TRACES_OSSERVATED_FOLDER_PATH
)
print(f"J_base = {J_base:.30e}")

# --- RIPRISTINO MODELLO BASE ---
shutil.copy(base_la_path, materials_paths['La'])
shutil.copy(base_mu_path, materials_paths['Mu'])

# --- PERTURBAZIONE DIAGNOSTICA ENORME ---
TEST_SCALE_LA = 1e9
TEST_SCALE_MU = 1e9

step_la = TEST_SCALE_LA * np.sign(dir_la)
step_mu = TEST_SCALE_MU * np.sign(dir_mu)

modify_h5_g(materials_paths['La'], step_la, mapping_list)
modify_h5_g(materials_paths['Mu'], step_mu, mapping_list)

# --- PULIZIA RICORSIVA DELLE TRACCE ---
all_old = glob.glob(os.path.join(TRACES_SIMULATED_FOLDER_PATH, "**", "*"), recursive=True)
removed = 0
for p in all_old:
    if os.path.isfile(p):
        os.remove(p)
        removed += 1

print("Removed old trace files:", removed)

# --- FORWARD ---
sbatch_and_wait(FORWARD_PROBLEM_SOLVER_SBATCH_PATH)

# --- ELENCO FILE NUOVI ---
all_new = sorted(
    [p for p in glob.glob(os.path.join(TRACES_SIMULATED_FOLDER_PATH, "**", "*"), recursive=True) if os.path.isfile(p)],
    key=os.path.getmtime
)

print("Number of new files:", len(all_new))
if all_new:
    print("Newest files:")
    for p in all_new[-5:]:
        print(" ", p, "mtime=", os.path.getmtime(p), "size=", os.path.getsize(p))

# --- MISFIT DOPO PERTURBAZIONE ---
J_test, _, _, _ = compute_misfit(
    TRACES_SIMULATED_FOLDER_PATH,
    TRACES_OSSERVATED_FOLDER_PATH
)

print(f"J_test = {J_test:.30e}")
print(f"DeltaJ = {(J_test - J_base):.30e}")

J_base = 1.377718529358251120520151289384e-01
Removed old trace files: 2
Submitted MESHER.sbatch from /usr/users/cea_seism/benede_gio/CEISM-Project/sem3d_config_files -> job 182136
Job 182136 finished with state: COMPLETED
MESHER.sbatch executed in 48.11 seconds

Submitted SOLVER.sbatch from /usr/users/cea_seism/benede_gio/CEISM-Project/sem3d_config_files -> job 182137
Job 182137 finished with state: COMPLETED
SOLVER.sbatch executed in 93.17 seconds

Number of new files: 2
Newest files:
  ./sem3d_config_files/traces/capteurs.0013.h5 mtime= 1775915834.1969097 size= 1061085
  ./sem3d_config_files/traces/capteurs.0023.h5 mtime= 1775915834.2209096 size= 4094542
J_test = 1.377718529358251120520151289384e-01
DeltaJ = 0.000000000000000000000000000000e+00


In [ ]:

    import shutil
    import glob

    '''
    ### ======================================== STEP 10 ======================================= ###    


    # 1. PRECONDIZIONAMENTO / SCALATURA DEL GRADIENTE
    # Imposta un limite massimo fisico per la perturbazione (es. 50 MPa o un valore appropriato per il tuo dominio)
    MAX_PERTURBATION = 5e7  

    max_dir_la = np.max(np.abs(dir_la)) + 1e-16 # Evita divisioni per zero
    max_dir_mu = np.max(np.abs(dir_mu)) + 1e-16

    scale_la = MAX_PERTURBATION / max_dir_la
    scale_mu = MAX_PERTURBATION / max_dir_mu

    # Applichiamo la scala ai vettori direzione
    dir_la = dir_la * scale_la
    dir_mu = dir_mu * scale_mu

    # Poiché p1 = grad_la * dir_la, se scaliamo dir_la di un fattore, anche p1 scala dello stesso fattore
    p1 = p1 * scale_la
    p2 = p2 * scale_mu

    # Inizializzazione Line Search
    alpha_la = 1.0 
    alpha_mu = 1.0
    c1 = 1e-4 
    xi = 0.5
    J_thresh = -np.inf
    J_learn = np.inf

    # 2. BACKUP DEI MODELLI BASE
    materials_paths = {
        'La': os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "example_la.h5"), 
        'Mu': os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "example_mu.h5")
    }

    base_la_path = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "base_la_backup.h5")
    base_mu_path = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "base_mu_backup.h5")

    # Salviamo lo stato m_0 intonso
    shutil.copy(materials_paths['La'], base_la_path)
    shutil.copy(materials_paths['Mu'], base_mu_path)

    mapping_list = [np.array([x_gl[i], y_gl[i], z_gl[i]]) for i in range(x_gl.size)]

    max_ls_iter = 10
    ls_iter = 0

    ## Backtracking loop ##
    while J_learn >= J_thresh:
        print(f"--- Line Search Iteration {ls_iter} ---")
        
        # 3. RIPRISTINO DEL MODELLO BASE PER APPLICARE L'UPDATE CORRETTO
        shutil.copy(base_la_path, materials_paths['La'])
        shutil.copy(base_mu_path, materials_paths['Mu'])

        # Calcoliamo la perturbazione per questo step (m_trial = m_0 + alpha * d)
        step_la = alpha_la * dir_la
        step_mu = alpha_mu * dir_mu

        # Applichiamo la perturbazione al file appena ripristinato
        modify_h5_g(materials_paths['La'], step_la, mapping_list)
        modify_h5_g(materials_paths['Mu'], step_mu, mapping_list)

        # 4. PULIZIA DEI VECCHI SISMOGRAMMI (Bug Override)
        simulated_traces = glob.glob(os.path.join(TRACES_SIMULATED_FOLDER_PATH, "*"))
        for trace_file in simulated_traces:
            if os.path.isfile(trace_file):
                os.remove(trace_file)

        # Eseguiamo il solver Forward
        sbatch_and_wait(FORWARD_PROBLEM_SOLVER_SBATCH_PATH)
        
        # Verifichiamo che i sismogrammi siano stati creati (Se SEM3D è crashato, la cartella è vuota)
        new_traces = glob.glob(os.path.join(TRACES_SIMULATED_FOLDER_PATH, "*"))
        if len(new_traces) == 0:
            print("ERRORE CRITICO: SEM3D non ha prodotto tracce. Parametri non fisici o instabilità CFL.")
            # Forziamo il backtracking punendo severamente J_learn
            J_learn = np.inf
        else:
            J_learn, _, _, _ = compute_misfit(TRACES_SIMULATED_FOLDER_PATH, TRACES_OSSERVATED_FOLDER_PATH)
        
        # Condizione di Armijo
        J_thresh = J + c1 * (alpha_la * p1 + alpha_mu * p2)
        
        print(f"J_base   = {J:.30e}")
        print(f"J_learn  = {J_learn:.30e}")
        print(f"J_thresh = {J_thresh:.12e}")
        print(f"alpha_la = {alpha_la:.12e} | alpha_mu = {alpha_mu:.30e}")
        print(f"diff     = {(J_learn - J_thresh):.30e}")
        print("--------------------------------------------------")

        if J_learn < J_thresh:
            print("Condizione di Armijo soddisfatta!")
            break

        # Riduzione del passo se non soddisfatto (o se c'è stato crash)
        alpha_la *= xi
        alpha_mu *= xi
        ls_iter += 1

    if ls_iter == max_ls_iter:
        print("ATTENZIONE: Raggiunto numero massimo iterazioni Line Search. Modello aggiornato con alpha minimo.")

    # 5. AGGIORNAMENTO MODELLO ADJOINTO
    materials_paths_adj = {
        'La': os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ, "example_la.h5"), 
        'Mu': os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ, "example_mu.h5")
    }

    # Il loop termina avendo trovato l'alpha corretto (o minimo), aggiorniamo la cartella adj
    # Nota: copio dal base dell'adjoin, altrimenti c'è lo stesso bug del forward!
    # Se i file .h5 in ADJ sono gli stessi di BASE prima dell'aggiornamento, possiamo 
    # sovrascriverli col risultato finale.
    shutil.copy(materials_paths['La'], materials_paths_adj['La'])
    shutil.copy(materials_paths['Mu'], materials_paths_adj['Mu'])

    '''
    ### ======================================== STEP 10 ======================================= ###    
    # Initialization of the Backtracking Line Search (BLS)

    alpha_la=1 
    alpha_mu=1
    c1=1e-4 
    xi=0.5
    J_thresh = -np.inf
    J_learn = np.inf
        
    vec_to_add_la, vec_to_add_mu = 2*alpha_la*dir_la , 2*alpha_mu*dir_mu
    materials_paths = {'La': os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH,"example_la.h5") , 'Mu' : os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH,"example_mu.h5")} # Enter the correct paths where the h5 material files are stored
    mapping_list = [np.array([x_gl[i],y_gl[i],z_gl[i]]) for i in range(x_gl.size)] # The mapping list that Long builds : [(x,y,z), (x,y,z), ...]    
    
    #test big perturbation
    TEST_SCALE_LA = 1e9
    TEST_SCALE_MU = 1e9
    vec_to_add_la = TEST_SCALE_LA * np.sign(dir_la)
    vec_to_add_mu = TEST_SCALE_MU * np.sign(dir_mu)
    
    modify_h5_g(materials_paths['La'],vec_to_add_la,mapping_list)
    modify_h5_g(materials_paths['Mu'],vec_to_add_mu,mapping_list)

    
    ls_iter = 0
    ## Backtracking loop ##
    while J_learn >= J_thresh:
        print(f"--- Line Search Iteration {ls_iter} ---")        
        vec_to_add_la -= alpha_la*dir_la
        vec_to_add_mu -= alpha_mu*dir_mu

        la_before = inspect_h5_samples(materials_paths['La'], "BEFORE")
        #mu_before = inspect_h5_samples(materials_paths['Mu'], "BEFORE")
        modify_h5_g(materials_paths['La'],vec_to_add_la,mapping_list)
        #print("la.h5 modified",flush=True)
        modify_h5_g(materials_paths['Mu'],vec_to_add_mu,mapping_list)
        #print("mu.h5 modified",flush=True)
        #la_after = inspect_h5_samples(materials_paths['La'], "AFTER")
        #mu_after = inspect_h5_samples(materials_paths['Mu'], "AFTER")

        #print("LA changed?", not np.array_equal(la_before, la_after))
        #print("MU changed?", not np.array_equal(mu_before, mu_after))

        #print("LA max abs diff =", np.max(np.abs(la_after - la_before)))
        #print("MU max abs diff =", np.max(np.abs(mu_after - mu_before)))

        #print("LA allclose?", np.allclose(la_before, la_after))
        #print("MU allclose?", np.allclose(mu_before, mu_after))

        # 4. PULIZIA DEI VECCHI SISMOGRAMMI (Bug Override)
        simulated_traces = glob.glob(os.path.join(TRACES_SIMULATED_FOLDER_PATH, "*"))
        for trace_file in simulated_traces:
            if os.path.isfile(trace_file):
                os.remove(trace_file)


        sbatch_and_wait(FORWARD_PROBLEM_SOLVER_SBATCH_PATH)

        # Verifichiamo che i sismogrammi siano stati creati (Se SEM3D è crashato, la cartella è vuota)
        new_traces = glob.glob(os.path.join(TRACES_SIMULATED_FOLDER_PATH, "*"))
        if len(new_traces) == 0:
            print("ERRORE CRITICO: SEM3D non ha prodotto tracce. Parametri non fisici o instabilità CFL.")
            # Forziamo il backtracking punendo severamente J_learn
            J_learn = np.inf
        else:
            J_learn, _, _, _ = compute_misfit(TRACES_SIMULATED_FOLDER_PATH, TRACES_OSSERVATED_FOLDER_PATH)

        J_thresh = J + c1*(alpha_la*p1 + alpha_mu*p2)

        print(f"J_base   = {J:.30e}")
        print(f"J_learn  = {J_learn:.30e}")
        print(f"J_thresh = {J_thresh:.12e}")
        print(f"alpha_la = {alpha_la:.12e} | alpha_mu = {alpha_mu:.30e}")
        print(f"diff     = {(J_learn - J_thresh):.30e}")
        print("--------------------------------------------------")

        if J_learn < J_thresh:
            print("Condizione di Armijo soddisfatta!")
            break

        alpha_la *= xi
        alpha_mu *= xi
        
        '''
        print(f"J        = {J:.12e}")
        print(f"J_learn  = {J_learn:.12e}")
        print(f"p1       = {p1:.12e}")
        print(f"p2       = {p2:.12e}")
        print(f"p1 + p2  = {(p1+p2):.12e}")
        print(f"alpha_la = {alpha_la:.12e}")
        print(f"alpha_mu = {alpha_mu:.12e}")

        print(f"||dir_la|| = {np.linalg.norm(dir_la):.12e}")
        print(f"||dir_mu|| = {np.linalg.norm(dir_mu):.12e}")
        print(f"dir_la min/max = {dir_la.min():.12e} / {dir_la.max():.12e}")
        print(f"dir_mu min/max = {dir_mu.min():.12e} / {dir_mu.max():.12e}")

        print(f"J_learn  = {J_learn:.12e}")
        print(f"J_thresh = {J_thresh:.12e}")
        print(f"diff     = {(J_learn - J_thresh):.12e}")
        '''
        ls_iter += 1
        print("--------------------------------------------------")
    materials_paths_adj = {'La': os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ,"example_la.h5") , 'Mu' : os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ,"example_mu.h5")} # Enter the correct paths where the h5 material files are stored    
    vec_to_add_la_adj = -alpha_la/xi*vec_to_add_la
    vec_to_add_mu_adj = -alpha_mu/xi*vec_to_add_mu
    modify_h5_g(materials_paths_adj['La'],vec_to_add_la_adj,mapping_list)
    modify_h5_g(materials_paths_adj['Mu'],vec_to_add_mu_adj,mapping_list)    

    '''
    mapping_list = load_global_xyz_tuples(LBFGS_OUTPUT_FOLDER_PATH)
    '''